In [1]:
# This code creates models for all of the rare animals.

import librosa
import numpy as np
import pandas as pd
import os
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [2]:
rare_amphibians = ['1176823', '1595929', '22930', '22956', '22983', '22985', '23150', '23154', '23176', '23724', '24285', '24287', '25073']
rare_amphibians += ['25214', '476521', '555123', '555145', '64898', '67107', '67252', '70711']
rare_birds = ['ashgre1', 'bafcur1', 'bcwfin2', 'bkhpar', 'blchaw1', 'blttit1', 'cibspi1', 'crebec1', 'dwatin1', 'fabwre1', 'ficman1', 'fotfly']
rare_birds += ['glteme1', 'grepot1', 'grhtan1', 'horscr1', 'hyamac1', 'lesgrf1', 'litcuc2', 'mabpar', 'magant1', 'magtan2', 'nacnig1', 'ocecra1']
rare_birds += ['orbtro3', 'palhor3', 'platyr1', 'pluibi1', 'purjay1', 'ragmac1', 'rivwar1', 'rufcac2', 'rufcas2', 'ruftho1', 'ruftof1']
rare_birds += ['ruther1', 'sabspa1', 'scther1', 'shshaw', 'smbtin1', 'souscr1', 'sptnig1', 'swthum1', 'whbant2', 'whlspi1', 'whnjay1', 'whwpic1']
rare_birds += ['yebcar', 'yecmac', 'yehcar1']
rare_insects = ['1161364', '47158son02', '47158son05', '47158son09', '47158son12', '47158son14', '47158son15', '47158son16', '47158son18']
rare_insects += ['47158son19', '47158son20', '760266']
rare_mammals = ['209233', '738183', '74580']

In [3]:
train_soundscapes_labels_df = pd.read_csv("train_soundscapes_labels.csv")

train_soundscapes_reduced_df = pd.DataFrame({
    "filename": train_soundscapes_labels_df.iloc[:, :2].astype(str).agg(''.join, axis=1),
    "primary_label": train_soundscapes_labels_df.iloc[:, 3]
})

In [4]:
SR = 32000
N_MELS = 128

rare_animals = rare_insects + rare_amphibians + rare_mammals + rare_birds
rare_animal_spectrogram_names = {}
rare_animal_spectrograms = {}

# Finds all of the labels for each rare class in train_soundscapes_reduced_df.

for row in train_soundscapes_reduced_df.itertuples():
    for animal in rare_animals:
        if animal in row.primary_label.split(";"):
            if animal not in rare_animal_spectrogram_names:
                rare_animal_spectrogram_names[animal] = []
            spectrogram = row.filename
            rare_animal_spectrogram_names[animal].append(spectrogram)

for animal in rare_animal_spectrogram_names:
    for filename in rare_animal_spectrogram_names[animal]:
        file = filename[:-8]
        start = int(filename[-2:])
    
        audio, _ = librosa.load(
            f"train_soundscapes/{file}",
            sr=SR,
            offset=start,
            duration=5
        )

# Converts the audio to a mel spectrogram, and normalizes it to the range [0, 1].

        mel = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=N_MELS)
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())
        if animal not in rare_animal_spectrograms:
            rare_animal_spectrograms[animal] = []
        rare_animal_spectrograms[animal].append(mel_db)

In [5]:
# Lists all of the folders in train_audio. Each folder corresponds to a different animal species. (Takes three minutes to run)
folders = [f for f in os.listdir(f"train_audio") if os.path.isdir(os.path.join(f"train_audio", f))]

chunk_length = 5 * SR

for animal in rare_animals:
    if animal in folders:
        folder = f"train_audio/{animal}"
        for filename in os.listdir(folder):
            audio, _ = librosa.load(
            f"{folder}/{filename}",
            sr=SR
            )
    
            # In animal_class_classifier, we ensured that all of the files in train_audio_df were at least five seconds long.
            audio_clips = []
            for i in range(int(len(audio)/chunk_length)):
                audio_clips += [audio[i*chunk_length:(i + 1)*chunk_length]]

            # Converts the audio files in audio_clips to a mel spectrogram, and normalizes it to the range [0, 1].

            for audio_clip in audio_clips:

                mel = librosa.feature.melspectrogram(
                    y=audio_clip,
                    sr=SR,
                    n_mels=N_MELS
                )

                mel_db = librosa.power_to_db(mel, ref=np.max)
                mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min())
                if animal not in rare_animal_spectrograms:
                    rare_animal_spectrograms[animal] = []
                rare_animal_spectrograms[animal].append(mel_db)

In [6]:
common_amphibians = ['1491113', '22961', '22967', '22973', '23158', '24279', '24321', '25092', '326272', '517063', '555146', '65377', '65380']
common_amphibians += ['66971']
common_birds = ['astcra1', 'baffal1', 'banana', 'barant1', 'batbel1', 'baymac', 'bbwduc', 'bkcdon', 'blheag1', 'bncfly', 'bobfly1', 'brcmar1']
common_birds += ['brnowl', 'bucmot4', 'bucpar', 'bufpar', 'bunibi1', 'burowl', 'camfli1', 'chacha1', 'chbmoc1', 'chobla1', 'chvcon1', 'coffal1']
common_birds += ['compau', 'compot1', 'crbthr1', 'epaori4', 'eulfly1', 'fepowl', 'flawar1', 'fusfly1', 'gilhum1', 'giwrai1', 'grasal3', 'greani1']
common_birds += ['greant1', 'greela', 'grekis', 'gretho2', 'greyel', 'grfdov1', 'gycwor1', 'houspa', 'larela1', 'lesela1', 'limpki', 'linwoo1']
common_birds += ['litnig1', 'masgna1', 'oliwoo1', 'orwpar', 'osprey', 'pabspi1', 'paltan1', 'phecuc1', 'picpig2', 'pirfly1', 'plasla1', 'plcjay1']
common_birds += ['pvttyr1', 'rebscy1', 'recfin1', 'redjun', 'relser1', 'rinkin1', 'roahaw', 'rubthr1', 'rufgna3', 'rufhor2', 'rufnig1', 'rumfly1']
common_birds += ['rutjac1', 'saffin', 'saytan1', 'scadov1', 'schpar1', 'shcfly1', 'shtnig1', 'sibtan2', 'smbani', 'sobcac1', 'sobtyr1']
common_birds += ['socfly1', 'sofspi1', 'souant1', 'soulap1', 'spbant3', 'spispi1', 'squcuc1', 'stbwoo2', 'strcuc1', 'strher2', 'strowl1']
common_birds += ['swtman1', 'tattin1', 'thlwre1', 'toctou1', 'trokin', 'trsowl', 'undtin1', 'varant1', 'watjac1', 'wesfie1', 'wfwduc1', 'whbwar2']
common_birds += ['whiwoo1', 'whtdov', 'y00678', 'yebela1', 'yecpar', 'yeofly1']
common_insects = ['47158son01', '47158son03', '47158son04', '47158son06', '47158son07', '47158son08', '47158son10', '47158son11', '47158son13']
common_insects += ['47158son17', '47158son21', '47158son22', '47158son23', '47158son24', '47158son25']
common_mammals = ['41970', '43435', '47144', '516975', '74113']

In [7]:
# For the rare classes, we start by finding the base rate that a given spectrogram is rare.
# For each rare animal, we already know the number of spectrograms containing that animal.
# We now need the total number of spectrograms in each class.

mammal_count = 0
amphibian_count = 0
insect_count = 0
bird_count = 0

rare_animal_counts = {animal: 0 for animal in rare_mammals + rare_amphibians + rare_insects + rare_birds}

for row in train_soundscapes_reduced_df.itertuples():
    animals = row.primary_label.split(";")
    for animal in animals:
        if animal in common_mammals or animal in rare_mammals:
            mammal_count += 1
        elif animal in common_amphibians or animal in rare_amphibians:
            amphibian_count += 1
        elif animal in common_insects or animal in rare_insects:
            insect_count += 1
        elif animal in common_birds or animal in rare_birds:
            bird_count += 1

for animal in rare_animal_counts:
    if rare_animal_counts[animal] == 0:
        rare_animal_counts[animal] = 1

rare_base_rates = {}

for mammal in rare_mammals:
    rare_base_rates[mammal] = len(rare_animal_spectrograms[mammal]) / mammal_count    
for amphibian in rare_amphibians:
    rare_base_rates[amphibian] = len(rare_animal_spectrograms[amphibian]) / amphibian_count    
for insect in rare_insects:
    rare_base_rates[insect] = len(rare_animal_spectrograms[insect]) / insect_count    
for bird in rare_birds:
    rare_base_rates[bird] = len(rare_animal_spectrograms[bird]) / bird_count

In [8]:
# For each rare animal, we train a model to classify whether a spectrogram contains that animal or not.
# Each class requires a separate model.

animal_model = tf.keras.models.load_model(
    f"best_multilabel_model.keras",
    compile=False
)

common_bird_model = tf.keras.models.load_model(f"best_common_bird_model_gpu.keras", compile=False)
common_amphibian_model = tf.keras.models.load_model(f"best_common_amphibian_model.keras", compile=False)
common_insect0_model = tf.keras.models.load_model(f"best_insect0_model.keras", compile=False)
common_insect1_model = tf.keras.models.load_model(f"best_insect1_model.keras", compile=False)
common_mammal_model = tf.keras.models.load_model(f"best_common_mammal_model.keras", compile=False)

from sklearn.metrics.pairwise import cosine_similarity

embeddings = {}

mammal_feature_model = tf.keras.Model(
    inputs=common_mammal_model.inputs,
    outputs=common_mammal_model.layers[-2].output
)

for mammal in rare_mammals:
    spec_array = np.array(rare_animal_spectrograms[mammal])
    rare_embeddings = mammal_feature_model.predict(spec_array)

    # Normalize for cosine similarity
    embeddings[mammal] = rare_embeddings / (
        np.linalg.norm(rare_embeddings, axis=1, keepdims=True) + 1e-8
    )
    
def rare_mammal_similarity_score(rare_mammal, new_spec):
    x = new_spec[np.newaxis, ...]

    x_emb = mammal_feature_model.predict(x)
    x_emb = x_emb / (np.linalg.norm(x_emb, axis=1, keepdims=True) + 1e-8)

    sims = cosine_similarity(x_emb, embeddings[rare_mammal])[0]

    return sims.max()

bird_feature_model = tf.keras.Model(
    inputs=common_bird_model.inputs,
    outputs=common_bird_model.layers[-2].output
)

for bird in rare_birds:
    spec_array = np.array(rare_animal_spectrograms[bird])
    rare_embeddings = bird_feature_model.predict(spec_array)

    # Normalize for cosine similarity
    embeddings[bird] = rare_embeddings / (
        np.linalg.norm(rare_embeddings, axis=1, keepdims=True) + 1e-8
    )
    
def rare_bird_similarity_score(rare_bird, new_spec):
    x = new_spec[np.newaxis, ...]

    x_emb = bird_feature_model.predict(x)
    x_emb = x_emb / (np.linalg.norm(x_emb, axis=1, keepdims=True) + 1e-8)

    sims = cosine_similarity(x_emb, embeddings[rare_bird])[0]

    return sims.max()

insect_feature_model = tf.keras.Model(
    inputs=common_insect1_model.inputs,
    outputs=common_insect1_model.layers[-2].output
)

for insect in rare_insects:
    spec_array = np.array(rare_animal_spectrograms[insect])
    rare_embeddings = insect_feature_model.predict(spec_array)

    # Normalize for cosine similarity
    embeddings[insect] = rare_embeddings / (
        np.linalg.norm(rare_embeddings, axis=1, keepdims=True) + 1e-8
    )
    
def rare_insect_similarity_score(rare_insect, new_spec):
    x = new_spec[np.newaxis, ...]

    x_emb = insect_feature_model.predict(x)
    x_emb = x_emb / (np.linalg.norm(x_emb, axis=1, keepdims=True) + 1e-8)

    sims = cosine_similarity(x_emb, embeddings[rare_insect])[0]

    return sims.max()

amphibian_feature_model = tf.keras.Model(
    inputs=common_amphibian_model.inputs,
    outputs=common_amphibian_model.layers[-2].output
)

for amphibian in rare_amphibians:
    spec_array = np.array(rare_animal_spectrograms[amphibian])
    rare_embeddings = amphibian_feature_model.predict(spec_array)

    # Normalize for cosine similarity
    embeddings[amphibian] = rare_embeddings / (
        np.linalg.norm(rare_embeddings, axis=1, keepdims=True) + 1e-8
    )
    
def rare_amphibian_similarity_score(rare_amphibian, new_spec):
    x = new_spec[np.newaxis, ...]

    x_emb = amphibian_feature_model.predict(x)
    x_emb = x_emb / (np.linalg.norm(x_emb, axis=1, keepdims=True) + 1e-8)

    sims = cosine_similarity(x_emb, embeddings[rare_amphibian])[0]

    return sims.max()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step 
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 17s 989ms/step
13/13 ━━━━━━━━━━━━━━━━━━━━ 12s 957ms/step
13/13 ━━━━━━━━━━━━━━━━━━━━ 12s 928ms/step
11/11 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step
10/10 ━━━━━━━━━━━━━━━━━━━━ 9s 877ms/step
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 947ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 14s 960ms/step
13/13 ━━━━━━━━━━━━━━━━━━━━ 12s 947ms/step
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 760ms/step
21/21 ━━━━━━━━━━━━━━━━━━━━ 18s 874ms/step
12/12 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step
18/18 ━━━━━━━━━━━━━━━━━━━━ 16s 901ms/step
20/20 ━━━━━━━━━━━━━━━━━━━━ 18s 892ms/step
15/15 ━━━━━━━━━━━━━━━━━━━━ 13s 896ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 15s 866ms/step
18/18 ━━━━━━━━━━━━━━━━━━━━ 16s 897ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 15s 891ms/step
8/8 ━━━━━━━━━━━━━━━━━━━━ 8s 1s/step
19/19 ━━━━━━━━━━━━━━━━━━━━ 19s 971ms/step
12/12 ━━━━━━━━━━━━━━━━━━━━ 10s 826ms/step
17/17 ━━━━━━━━━━━━━━━━━━━━ 16s 931ms/step
12/12 ━━━━━━━━━━━━━

In [9]:
embeddings = {animal: [] for animal in rare_amphibians + rare_insects + rare_mammals + rare_birds}

for bird in rare_birds:
    for mel_spec in rare_animal_spectrograms[bird]:
        embedding = bird_feature_model.predict(np.array([mel_spec]),verbose=0)[0]
        embeddings[bird].append(embedding)
for insect in rare_insects:
    for mel_spec in rare_animal_spectrograms[insect]:
        embedding = insect_feature_model.predict(np.array([mel_spec]),verbose=0)[0]
        embeddings[insect].append(embedding)
for mammal in rare_mammals:
    for mel_spec in rare_animal_spectrograms[mammal]:
        embedding = mammal_feature_model.predict(np.array([mel_spec]),verbose=0)[0]
        embeddings[mammal].append(embedding)
for amphibian in rare_amphibians:
    for mel_spec in rare_animal_spectrograms[amphibian]:
        embedding = amphibian_feature_model.predict(np.array([mel_spec]),verbose=0)[0]
        embeddings[amphibian].append(embedding)

for species in embeddings:
    embeddings[species] = np.array(embeddings[species])

In [10]:
def estimate_species_center(species, percentile=25):
    E = embeddings[species]

    E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-8)

    sims = E @ E.T

    # Remove self-similarity values of 1
    mask = ~np.eye(len(E), dtype=bool)
    pair_sims = sims[mask]

    if len(pair_sims) == 0:
        return 0.90

    return float(np.percentile(pair_sims, percentile))

In [12]:
rare_centers = {}

rare_animals = rare_mammals + rare_amphibians + rare_insects + rare_birds

for species in rare_animals:
    rare_centers[species] = estimate_species_center(species, percentile=25)

In [13]:
def estimate_species_sharpness(species):
    E = embeddings[species]

    E = E / (np.linalg.norm(E, axis=1, keepdims=True) + 1e-8)

    sims = E @ E.T
    mask = ~np.eye(len(E), dtype=bool)
    pair_sims = sims[mask]

    if len(pair_sims) < 3:
        return 20

    spread = np.std(pair_sims)

    sharpness = 1 / (spread + 1e-6)

    return float(np.clip(sharpness, 10, 60))

In [15]:
rare_sharpnesses = {}

for species in rare_animals:
    rare_sharpnesses[species] = estimate_species_sharpness(species)

In [14]:
import pickle

# Open a file in write-binary mode
with open('rare_embeddings.pkl', 'wb') as f:
    pickle.dump(embeddings, f)
    
with open('rare_base_rates.pkl', 'wb') as f:
    pickle.dump(rare_base_rates, f)

In [16]:
import pickle

with open("rare_centers.pkl", "wb") as f:
    pickle.dump(rare_centers, f)

with open("rare_sharpnesses.pkl", "wb") as f:
    pickle.dump(rare_sharpnesses, f)